### Feature Engineering - Aviation Accident Modeling

This notebook prepares features for machine learning models to predict damage types. It performs comprehensive data preprocessing and feature engineering for aviation safety analysis and transforms raw accident data into machine learning-ready features for predictive modeling. 

The pipeline includes data cleaning, categorical encoding, numerical transformations, and feature selection. Output features can be used to build models for accident severity classification and damage assessment. The notebook also handles missing data imputation and creates derived features from temporal and categorical variables. 

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load cleaned data
df = pd.read_csv('data\CleanedAviationData.csv')
print(f"Dataset shape: {df.shape}")

Dataset shape: (88882, 22)


## Feature Selection and Engineering

In [4]:
# Selecting relevant features for modeling
feature_columns = [
    'Make', 'Model', 'Engine.Type',
    'Weather.Condition', 'Broad.phase.of.flight',
    'Purpose.of.flight', 'Amateur.Built', 'Month', 'Year'
]

# Target variables
target_columns = {
    # 'severity': 'Severity_Category',
    # 'fatal': 'Is_Fatal',
    'damage': 'Aircraft.damage'
}

# Create modeling dataset
model_df = df[feature_columns + list(target_columns.values())].copy()

# Remove rows with missing target values
model_df = model_df.dropna(subset=['Aircraft.damage'])

print(f"Modeling dataset shape: {model_df.shape}")
print(f"Missing values:\n{model_df.isnull().sum()}")

Modeling dataset shape: (88882, 10)
Missing values:
Make                     0
Model                    0
Engine.Type              0
Weather.Condition        0
Broad.phase.of.flight    0
Purpose.of.flight        0
Amateur.Built            0
Month                    0
Year                     0
Aircraft.damage          0
dtype: int64


## Categorical Encoding

In [5]:
# Encode categorical variables
categorical_features = ['Make', 'Model', 'Engine.Type', 'Weather.Condition', 
                       'Broad.phase.of.flight', 'Purpose.of.flight', 'Amateur.Built']

# Label encode categorical features
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    model_df[col] = model_df[col].fillna('Unknown')
    model_df[col + '_encoded'] = le.fit_transform(model_df[col])
    label_encoders[col] = le

# Encode target variables
target_encoders = {}
for target_name, target_col in target_columns.items():
    # if target_col != 'Is_Fatal':  # Already binary
        le = LabelEncoder()
        model_df[target_col + '_encoded'] = le.fit_transform(model_df[target_col])
        target_encoders[target_name] = le

print("Categorical encoding completed")

Categorical encoding completed


## Feature Matrix Creation

In [10]:
# Create feature matrix
feature_cols_encoded = [col + '_encoded' for col in categorical_features] + \
                      ['Month', 'Year']

X = model_df[feature_cols_encoded].fillna(0)

# Target variables
y_damage = model_df['Aircraft.damage_encoded']

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {feature_cols_encoded}")
print(f"\nTarget distributions:")
print(f"Damage: {model_df['Aircraft.damage'].value_counts()}")

Feature matrix shape: (88882, 9)
Features: ['Make_encoded', 'Model_encoded', 'Engine.Type_encoded', 'Weather.Condition_encoded', 'Broad.phase.of.flight_encoded', 'Purpose.of.flight_encoded', 'Amateur.Built_encoded', 'Month', 'Year']

Target distributions:
Damage: Substantial    64147
Destroyed      18617
Unknown         3313
Minor           2805
Name: Aircraft.damage, dtype: int64


## Train-Test Split

In [11]:
# Split data for each target
test_size = 0.2
random_state = 42

# Damage prediction split
X_train_dmg, X_test_dmg, y_train_dmg, y_test_dmg = train_test_split(
    X, y_damage, test_size=test_size, random_state=random_state, stratify=y_damage
)

print(f"Training set sizes:")
print(f"Damage: {X_train_dmg.shape[0]}")

Training set sizes:
Damage: 71105


## Save Processed Data

In [16]:
# Save feature-engineered data
model_df.to_csv("data/aviation_features.csv", index=False)

# Save train-test splits
import pickle

splits_data = {
    'damage': (X_train_dmg, X_test_dmg, y_train_dmg, y_test_dmg),
    'encoders': {'label_encoders': label_encoders, 'target_encoders': target_encoders},
    'feature_names': feature_cols_encoded
}

with open('data/model_damaged.pkl', 'wb') as f:
    pickle.dump(splits_data, f)

print("Feature engineering completed and saved!")
print(f"Features ready for modeling: {len(feature_cols_encoded)}")
print(f"Total samples: {len(X):,}")

Feature engineering completed and saved!
Features ready for modeling: 9
Total samples: 88,882
